# RemBERT — locking SentencePiece superiority (2nd SP encoder)

**Goal.** The flip matrix left a seed-stable finding: **QA ≈ BIO under WordPiece**
(mBERT, MuRIL), **QA wins big under SentencePiece** (XLM-R). The Main blocker is
that SentencePiece is **n=1** (XLM-R only), so the claim reads *"XLM-R amplifies QA"*
not *"SentencePiece amplifies QA"*. This notebook adds **`google/rembert`** — a 2nd
SentencePiece encoder, with a *different* tokenizer AND a *different* architecture
from XLM-R — to decide:

- RemBERT shows the large QA gain too → **"SentencePiece amplifies QA"** (Main).
- RemBERT does not → XLM-R-specific; claim narrows.

> **Why not mDeBERTa-v3?** The original target was `microsoft/mdeberta-v3-base`, but
> its disentangled attention is numerically unstable to fine-tune on a T4 (NaN loss
> within epoch 1, unrecoverable — see git history). RemBERT uses standard attention,
> fine-tunes cleanly, and its different architecture makes the "it's the tokenizer,
> not the model" argument *stronger*, not weaker.

**Two proofs, cheap first:**
1. **Mechanism (no GPU, §3):** tokenizer reachability. SentencePiece pushes gold spans
   off word boundaries → high `subtok%` (QA-only-reachable). If RemBERT's SP tokenizer
   patterns with XLM-R (not mBERT/MuRIL), that alone predicts QA>BIO.
2. **Empirical (GPU, §5):** train QA + BIO under RemBERT × 3 seeds; measure the QA−BIO
   exact gap and drop it next to XLM-R / mBERT / MuRIL.

> **Metric integrity.** All numbers here are **FRESH — not canonical, not in
> `key_numbers.md`.** They must clear `Evaluation/Full_evaluation.py` before any paper
> use. Nothing below hardcodes a metric; every value is computed live.

> **Batch size.** RemBERT (~576M params) uses batch 16 on the T4 (the XLM-R/mBERT/MuRIL
> matrix used 32). The QA−BIO gap is measured *within* RemBERT (QA and BIO both at 16),
> and the cross-encoder comparison is of *gaps*, so this is not a confound.

## 0. GPU check (set Runtime → T4 GPU first)

In [ ]:
import os
assert os.system('nvidia-smi >/dev/null 2>&1') == 0, 'No GPU — Runtime → Change runtime type → T4 GPU'
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Clone / pull repo + SentencePiece deps

In [ ]:
from pathlib import Path
REPO = '/content/Idiomator_Research'
if Path(REPO).exists():
    !cd $REPO && git pull --ff-only
else:
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git $REPO
%cd $REPO
!pip install -q -r Requirements.txt
# SentencePiece fast tokenizers need sentencepiece + protobuf:
!pip install -q sentencepiece protobuf
# Smoke-load the tokenizer + model now so a download/conversion failure surfaces
# here, not 20 min into a training run.
from transformers import AutoTokenizer, AutoModel
MODEL = 'google/rembert'
_tk = AutoTokenizer.from_pretrained(MODEL)
_enc = _tk('idiom check', return_offsets_mapping=True)
assert 'offset_mapping' in _enc, 'RemBERT tokenizer did not return offsets — span alignment will fail'
_m = AutoModel.from_pretrained(MODEL)
print(f'OK: {MODEL} | type_vocab_size={_m.config.type_vocab_size} | hidden={_m.config.hidden_size} | params={sum(p.numel() for p in _m.parameters())/1e6:.0f}M')
del _m  # free RAM before training

## 2. Config — mount Drive + persistence gate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

FORCE = False        # <-- True to overwrite existing runs (post code-fix rerun)

DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
OUT_BASE  = f'{DRIVE_OUT}/rembert'       # RemBERT runs land here
FLIP_DIR  = f'{DRIVE_OUT}/flip'          # existing XLM-R/mBERT/MuRIL runs (for comparison)
os.environ['DRIVE_OUT'] = DRIVE_OUT
os.environ['FORCE'] = '1' if FORCE else '0'
Path(OUT_BASE).mkdir(parents=True, exist_ok=True)

# Persistence gate (CLAUDE.md hard rule): outputs MUST be on Drive, and Drive
# must be writable+readable-back, before any training cell runs.
assert OUT_BASE.startswith('/content/drive/'), 'persistence gate: output not on Drive'
_probe = Path(OUT_BASE) / '.persist_probe'
_probe.write_text('ok'); assert _probe.read_text() == 'ok', 'Drive readback failed'
_probe.unlink()
print(f'✓ persistence gate passed | OUT_BASE = {OUT_BASE} | FORCE = {FORCE}')
print(f'  XLM-R/mBERT/MuRIL comparison dir: {FLIP_DIR} (present: {Path(FLIP_DIR).exists()})')

## 3. Mechanism FIRST (no GPU) — SentencePiece reachability

Per language, classify each gold span by the tightest unit a tokenizer can represent it with:
**word%** (BIO can reach), **subtok%** (only QA can reach — a sub-word boundary), **none%**.
Higher `subtok%` under a tokenizer predicts **QA > BIO** for that tokenizer. The test: does
RemBERT's SentencePiece tokenizer sit with **XLM-R (SP)**, away from **mBERT/MuRIL (WordPiece)**?

In [ ]:
import json, statistics
from collections import defaultdict
from transformers import AutoTokenizer
load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
LANGS_ORDER = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']
GOLD = 'idioms_structured/Splits/test.jsonl'
gold = [r for r in load(GOLD) if r.get('span_start') is not None]

# Reachability via char_to_token + word_ids (Rust-backed, robust across tokenizers).
# NOTE: an earlier version reconstructed boundaries from offset_mapping; that broke
# for SentencePiece tokenizers with a leading-space marker (▁) — RemBERT/mDeBERTa
# reported ~80% 'none' purely as an offset-convention artifact. This version uses
# the same char_to_token API the trainers use, and reproduces the offset method
# EXACTLY on XLM-R and mBERT (validated) while working correctly for RemBERT.
def reachability(tok_name):
    tk = AutoTokenizer.from_pretrained(tok_name)
    per = defaultdict(lambda: dict(word=0, subtok=0, none=0, n=0))
    for r in gold:
        s = r['sentence']; gs, ge = r['span_start'], r['span_end']; lang = r['language']
        enc = tk(s)
        wids = enc.word_ids(); N = len(wids)
        t_start = enc.char_to_token(gs)      # token holding the gold start char
        t_end   = enc.char_to_token(ge - 1)  # token holding the gold last char
        c = per[lang]; c['n'] += 1
        if t_start is None or t_end is None or wids[t_start] is None or wids[t_end] is None:
            c['none'] += 1; continue
        ws, we = wids[t_start], wids[t_end]
        first_of_word = (t_start == 0)   or (wids[t_start - 1] != ws)  # gold starts at a word start
        last_of_word  = (t_end == N - 1) or (wids[t_end + 1]   != we)  # gold ends at a word end
        if first_of_word and last_of_word:
            c['word'] += 1      # whole-word span → BIO (word-level) can reach it
        else:
            c['subtok'] += 1    # boundary cuts inside a word → only QA-style can reach it
    return per

# family-grouped so the SP-vs-WordPiece split is visible
TOKS = {
    'xlmr (SP)':      'xlm-roberta-base',
    'rembert (SP)':   'google/rembert',
    'mbert (WP)':     'bert-base-multilingual-cased',
    'muril (WP)':     'google/muril-base-cased',
}
print(f"{'tok':16}{'lang':11}{'word%(BIO-ok)':>14}{'subtok%(QA-only)':>18}{'none%':>8}")
sp_sub, wp_sub = [], []
for name, hf in TOKS.items():
    per = reachability(hf)
    for lang in LANGS_ORDER:
        c = per.get(lang)
        if not c or c['n'] == 0: continue
        n = c['n']; sub = 100*c['subtok']/n
        (sp_sub if '(SP)' in name else wp_sub).append(sub)
        print(f"{name:16}{lang:11}{100*c['word']/n:>13.1f}{sub:>18.1f}{100*c['none']/n:>8.1f}")
    print()
print(f"MEAN subtok% (QA-only-reachable):  SentencePiece = {statistics.mean(sp_sub):.1f}   "
      f"WordPiece = {statistics.mean(wp_sub):.1f}")
print('Read: if rembert(SP) subtok%% tracks XLM-R(SP) and sits ABOVE WordPiece, the mechanism')
print('      for "SentencePiece amplifies QA" replicates on a 2nd SP tokenizer — before any training.')

## 4. Compatibility dry-run gate (1 epoch English, joint + BIO)

Runs `run_07_mdeberta_sp_replication.sh --dry-run` — the single source of truth for the
training commands (now defaults to RemBERT; `MODEL`/`ENC_SHORT` env vars override it).
It trains 1 English epoch for **both** trainers and asserts each produced
`test_predictions.jsonl`. **Do not run §5 unless this cell prints `DRY-RUN PASSED`.**

In [ ]:
import subprocess, sys

def _run_streaming(cmd):
    """Run shell command, stream stdout+stderr to cell, return exit code."""
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

rc = _run_streaming('bash Additional_Rigor_Experiments/run_07_mdeberta_sp_replication.sh --dry-run')
assert rc == 0, 'DRY-RUN FAILED — RemBERT incompatible with a trainer; inspect the console.log paths printed above. DO NOT launch §5.'
print('\n✓ DRY-RUN PASSED — RemBERT works with both trainers. Cleaning dry dirs, proceed to §5.')
import shutil
for d in ('_dryrun_joint', '_dryrun_bio'):
    shutil.rmtree(f'{OUT_BASE}/{d}', ignore_errors=True)


## 5. Full matrix — RemBERT QA vs BIO × seeds {42, 123, 7}

Calls the same script (no `--dry-run`). It skips runs with an existing `metrics.json`
unless `FORCE=1`, tees per-run `console.log` to Drive incrementally, and deletes
`best_model/` after each run (checkpoint policy + Drive quota). ~6 runs; long — keep
the tab alive and watch §6 from a second tab.

In [ ]:
import subprocess, sys

def _run_streaming(cmd):
    """Run shell command, stream stdout+stderr to cell, return exit code."""
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

rc = _run_streaming('bash Additional_Rigor_Experiments/run_07_mdeberta_sp_replication.sh')
print('exit:', rc)


## 6. Live monitor (run in a SEPARATE Colab tab)

In [ ]:
# Polls the Drive folder; self-contained (mounts Drive). Loops until 6 jobs done.
import os, time
from pathlib import Path
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception:
    pass
from IPython.display import clear_output

OUT_BASE = os.environ.get('DRIVE_OUT', '/content/drive/MyDrive/IdiomatorRigor') + '/rembert'
SYST, SEEDS = ['joint', 'bio'], [42, 123, 7]
EXPECT = [(s, sd) for s in SYST for sd in SEEDS]   # 6
REFRESH = 20

def st(s, sd):
    d = Path(f'{OUT_BASE}/rembert_{s}_s{sd}')
    if (d / 'metrics.json').exists():
        return '✓'
    log = d / 'console.log'
    if log.exists() and time.time() - log.stat().st_mtime < 180:
        return '~'
    if log.exists():
        return 'x'
    return '.'

while True:
    marks = {j: st(*j) for j in EXPECT}
    done = sum(v == '✓' for v in marks.values()); run = sum(v == '~' for v in marks.values())
    clear_output(wait=True)
    fill = int(34 * done / len(EXPECT))
    print(f'RemBERT QA-vs-BIO   {done}/{len(EXPECT)} done, {run} running')
    print('[' + '#'*fill + '-'*(34-fill) + ']\n')
    print(f"{'system':8}" + ''.join(f's{sd:<5}' for sd in SEEDS))
    for s in SYST:
        print(f'{s:8}' + ''.join(f'{marks[(s, sd)]:<6}' for sd in SEEDS))
    print('\nlegend: ✓ done   ~ running   x stale/disconnected   . pending')
    if done == len(EXPECT):
        print('\nALL 6 DONE — run §7 aggregation.'); break
    time.sleep(REFRESH)

## 7. Aggregate — SentencePiece n=2: RemBERT vs XLM-R vs WordPiece

QA−BIO **exact** gap per (encoder, language), mean ± seed std. RemBERT is read from
`OUT_BASE`; XLM-R / mBERT / MuRIL from the existing `FLIP_DIR` if present. The family
means at the bottom are the headline: **SentencePiece gap (XLM-R + RemBERT) vs WordPiece
gap (mBERT + MuRIL)**. All computed live — no hardcoded metrics.

In [ ]:
import json, statistics
from collections import defaultdict
load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8')]
LANGS_ORDER = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

def exact_by_lang(path):
    agg = defaultdict(list)
    for r in load(path):
        gs, ge = r.get('span_start'), r.get('span_end')
        ps, pe = r.get('pred_span_start'), r.get('pred_span_end')
        agg[r['language']].append(int(ps is not None and gs is not None and ps == gs and pe == ge))
    return {l: sum(v)/len(v) for l, v in agg.items()}

# enc -> (dir, name-template, family). RemBERT from OUT_BASE; others from FLIP_DIR.
SOURCES = {
    'rembert':  (OUT_BASE, 'rembert_{system}_s{seed}',  'SentencePiece'),
    'xlmr':     (FLIP_DIR, 'xlmr_{system}_s{seed}',     'SentencePiece'),
    'mbert':    (FLIP_DIR, 'mbert_{system}_s{seed}',    'WordPiece'),
    'muril':    (FLIP_DIR, 'muril_{system}_s{seed}',    'WordPiece'),
}
SEEDS = [42, 123, 7]

cell = defaultdict(lambda: defaultdict(list))   # (enc, system) -> lang -> [exact per seed]
present = []
for enc, (base, tmpl, fam) in SOURCES.items():
    has_any = False
    for system in ('joint', 'bio'):
        for seed in SEEDS:
            p = Path(base) / tmpl.format(system=system, seed=seed) / 'test_predictions.jsonl'
            if not p.exists():
                continue
            has_any = True
            for lang, ex in exact_by_lang(p).items():
                cell[(enc, system)][lang].append(ex)
    if has_any: present.append(enc)
missing = [e for e in SOURCES if e not in present]
if missing:
    print(f'NOTE: no runs found for {missing} (FLIP_DIR not populated?) — showing what is present.\n')

print(f"{'family':14}{'enc':10}{'lang':11}{'QA':>6}{'BIO':>7}{'gap':>8}{'±std':>7}  winner")
fam_gaps = defaultdict(list)   # family -> per-encoder mean gap (across langs)
for enc in present:
    fam = SOURCES[enc][2]
    enc_lang_gaps = []
    for lang in LANGS_ORDER:
        qa = cell[(enc,'joint')].get(lang); bio = cell[(enc,'bio')].get(lang)
        if not qa or not bio: continue
        qm, bm = statistics.mean(qa), statistics.mean(bio)
        gaps = [a-b for a, b in zip(qa, bio)]
        g = statistics.mean(gaps); sd = statistics.pstdev(gaps) if len(gaps) > 1 else 0.0
        enc_lang_gaps.append(g)
        print(f"{fam:14}{enc:10}{lang:11}{qm:>6.2f}{bm:>7.2f}{g:>+8.2f}{sd:>7.2f}  {'QA' if g>0 else 'BIO'}")
    if enc_lang_gaps:
        fam_gaps[fam].append(statistics.mean(enc_lang_gaps))
    print()

print('── HEADLINE: mean QA−BIO gap by tokenizer family ──')
for fam in ('SentencePiece', 'WordPiece'):
    vals = fam_gaps.get(fam, [])
    if vals:
        encs = [e for e in present if SOURCES[e][2] == fam]
        print(f"  {fam:14} mean gap = {statistics.mean(vals):+.2f}   (encoders: {', '.join(encs)}, n={len(vals)})")
if fam_gaps.get('SentencePiece') and fam_gaps.get('WordPiece'):
    sp, wp = statistics.mean(fam_gaps['SentencePiece']), statistics.mean(fam_gaps['WordPiece'])
    ratio = sp/wp if wp else float('inf')
    print(f"\n  SentencePiece gap is {ratio:.1f}× the WordPiece gap.")
    if len(fam_gaps['SentencePiece']) >= 2:
        print('  → 2 SP encoders agree: claim upgrades to "SentencePiece amplifies QA" (Main-ready).')
    else:
        print('  → still only 1 SP encoder with results — RemBERT runs incomplete.')
print('\nREMINDER: FRESH runs — NOT canonical. Clear Full_evaluation.py before any paper use.')

## 8. Decisive test — extended-gold (run_08), pure CPU, no retrain

Re-scores the **existing seed-42 predictions** under three gold normalisations to decide whether the SentencePiece QA−BIO exact-match gap is a *trailing-punctuation artifact*:

- `original` — gold as-is.
- `extend`   — grow `span_end` through trailing punct (rewards BIO).
- `strip`    — strip trailing punct from **both** gold and prediction (symmetric, fair).

WordPiece encoders (mBERT, MuRIL) are the null control — their gap should stay ≈0 in every mode. Reports per-language QA/BIO exact, the paired gap, and bootstrap CIs, then a per-family verdict (Scenario A: artifact / Scenario B: genuine SP advantage).

Runs in seconds on CPU. Seed-42 only by default; once s123/s7 land, re-run with `--seeds 42 123 7`.

In [ ]:
# §8 — extended-gold decisive scorer. Files already live on the mounted Drive;
# the script reads them in place (no upload, no GPU).
!cd /content/Idiomator_Research && python Additional_Rigor_Experiments/run_08_extended_gold.py \
    --preds-root /content/drive/MyDrive/IdiomatorRigor --force
# When seeds 123 & 7 finish, add:  --seeds 42 123 7

## Next steps

- **If RemBERT replicates the large SP gap** (and §3 mechanism agrees): SentencePiece is now
  **n=2** → upgrade the IdiomBERT Main claim from *"XLM-R amplifies QA"* to
  *"SentencePiece amplifies QA"*. Then run the EN+ES+HI+TE runs through
  `Evaluation/Full_evaluation.py` to promote any number from fresh → canonical before drafting.
- **If it does not**: the effect is XLM-R-specific; narrow the claim and lean on MultiIdiom as the
  primary Main bet (per HANDOFF portfolio).
- Either way, fold the RemBERT column into the flip table and update `HANDOFF.md`.